# Module 6 — Running it

Modules 1–5 were about *what the model does*. This one is about **operating it**: the two
entry points, what data goes where, the gates, and how to read a run.

If you own this model day to day, this is the module that matters most.

## The one idea underneath all of it

> **The model is ten numbers. Everything in `data/raw/` exists to produce them; once they
> exist, pricing a book needs none of it.**

Those ten numbers — five for the VIX response, five for vol-of-vol — are the calibration.
Producing them is slow and should be a reviewed event. *Using* them is fast and happens
whenever a book arrives. Keeping that line clear is what makes the model a reviewable
object rather than something that re-derives itself on every run.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd()
while not (REPO / "vixshock").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
pd.set_option("display.width", 200)

import config
print(f"repo: {REPO}")

repo: C:\Users\vmc30\OneDrive\Desktop\Personal_Repos\Work\VIX


## Step 1 — the two entry points

```
python run.py                          CALIBRATE + validate + price.  Minutes.  YEARLY.
python price.py --book input/b.csv     PRICE ONLY, frozen calibration.  ~1 second.  DAILY.
```

`run.py` refits both responses from 22 years of history, re-runs the 14 stress episodes,
checks every gate, and rewrites the parameter files. `price.py` loads those parameters
and applies them to today's curve. It never refits.

**Why the split exists** — this is the argument to have ready:

1. Refitting on every run makes the risk number move for reasons unrelated to the book
2. Two runs on the same book days apart disagree, with nothing recording why
3. There is no attribution: book change or calibration change?
4. **A validator signs off on *a model*. If it re-derives itself every run, there is no
   fixed object to sign off on.** That is the decisive one.

## Step 2 — what each entry point actually reads

In [2]:
rows = [
    ("VIX futures curve", "data/raw/vx/*.csv", "FRESH", "yes", "yes"),
    ("spot VIX", "data/raw/vix.csv", "FRESH", "yes", "yes"),
    ("vol-of-vol level", "data/processed/vov_levels.csv", "FRESH", "yes", "yes"),
    ("the ten fitted numbers", "output/*_params.json", "FROZEN", "writes them", "reads them"),
    ("SPX history", "data/raw/spx.csv", "calibration only", "yes", "NO"),
    ("VVIX / historical IV", "data/raw/, bloomberg_historical/", "calibration only", "yes", "NO"),
    ("the book", "input/book_<date>.csv", "every run", "optional", "yes"),
]
print("WHAT EACH ENTRY POINT NEEDS\n")
print(pd.DataFrame(rows, columns=["input", "where", "kind", "run.py", "price.py"]).to_string(index=False))

print("\n\nTHE POINT MOST PEOPLE MISS:")
print("  SPX is NOT a pricing input. The SPX move is a SCENARIO YOU CHOOSE (-20%, -10%),")
print("  not something observed. shock.py reads no SPX data at all.")
print("\n  This is why the Yahoo SPX pull is not a daily dependency: it is touched only")
print("  during a recalibration, where a qualified vendor source can be substituted.")

WHAT EACH ENTRY POINT NEEDS

                 input                            where             kind      run.py   price.py
     VIX futures curve                data/raw/vx/*.csv            FRESH         yes        yes
              spot VIX                 data/raw/vix.csv            FRESH         yes        yes
      vol-of-vol level    data/processed/vov_levels.csv            FRESH         yes        yes
the ten fitted numbers             output/*_params.json           FROZEN writes them reads them
           SPX history                 data/raw/spx.csv calibration only         yes         NO
  VVIX / historical IV data/raw/, bloomberg_historical/ calibration only         yes         NO
              the book            input/book_<date>.csv        every run    optional        yes


THE POINT MOST PEOPLE MISS:
  SPX is NOT a pricing input. The SPX move is a SCENARIO YOU CHOOSE (-20%, -10%),
  not something observed. shock.py reads no SPX data at all.

  This is why the Yahoo SPX p

## Step 3 — the calibration in force

Every `price.py` run prints this at the top. It is the provenance record: *which* ten
numbers priced this book, fitted over what window, written when.

In [3]:
import datetime as dt

for name, fn in (("VIX response", "response_params.json"), ("vol-of-vol", "vov_params.json")):
    p = REPO / "output" / fn
    if not p.exists():
        print(f"{name}: MISSING - run `python run.py` once to produce it")
        continue
    d = json.loads(p.read_text())
    age = (dt.date.today() - dt.datetime.fromtimestamp(p.stat().st_mtime).date()).days
    print(f"{name}")
    print(f"   fitted     {d['fit_start']} -> {d['fit_end']}   n = {d['n_obs']:,}")
    print(f"   method     {d['method_down']}  q={d['quantile']}")
    print(f"   written    {dt.datetime.fromtimestamp(p.stat().st_mtime):%Y-%m-%d %H:%M}  ({age} days ago)")
    print(f"   the five:  " + "  ".join(f"{k}={d[k]:.4g}" for k in
                                        ("beta_0", "k", "lam", "beta_up_0", "lam_up")))
    print()

print(f"Recalibration policy: YEARLY.  price.py FAILS beyond")
print(f"config.MAX_CALIBRATION_AGE_DAYS = {getattr(config,'MAX_CALIBRATION_AGE_DAYS','(not set)')} days.")

VIX response
   fitted     2004-03-29 -> 2026-09-18   n = 112,793
   method     envelope  q=0.95
   written    2026-09-19 11:13  (1 days ago)
   the five:  beta_0=197.5  k=0.7586  lam=0.209  beta_up_0=68.26  lam_up=0.2179

vol-of-vol
   fitted     2006-04-25 -> 2026-09-16   n = 69,419
   method     envelope  q=0.95
   written    2026-09-19 11:14  (1 days ago)
   the five:  beta_0=1203  k=-10.26  lam=0.1813  beta_up_0=301.5  lam_up=0.3436

Recalibration policy: YEARLY.  price.py FAILS beyond
config.MAX_CALIBRATION_AGE_DAYS = 400 days.


## Step 4 — the gates

The model refuses to bless itself. Both entry points exit non-zero if any gate fails.
**A non-zero exit means do not use the numbers.**

### `run.py` gates — is the *model* sound?

| gate | fails when | what it protects |
|---|---|---|
| `cm_vs_spot` | CM-30 vs spot correlation < 0.90, or inversion < 50% of high-VIX days | the curve construction (Module 1) |
| `params_in_range` | any fitted parameter outside `config.PARAM_RANGES` | a fit that has gone somewhere strange |
| `stress` | > 2 understated episodes per tenor, or any ratio < 0.5 | the shock is big enough (Module 4) |
| `vov_params_in_range` | same, for the vol-of-vol layer | Module 5 |
| `data_fresh` | the curve is older than `MAX_DATA_AGE_DAYS` | silent staleness |

### `price.py` gates — is *this run* usable?

| gate | fails when | what to do |
|---|---|---|
| `calibration_fresh` | parameters older than `MAX_CALIBRATION_AGE_DAYS` (400) | recalibrate — a reviewed event |
| `data_fresh` | curve older than `MAX_DATA_AGE_DAYS` (7) | refresh, or use `data/daily_inputs/` |
| `book_vols` | **any position fell back to the ATM curve** | add premiums — see `input/INPUT_CONTRACT.md` |

`book_vols` is the far-OTM protection. Without it, a book missing premiums marks
out-of-the-money calls near zero and still reports a clean run.

In [4]:
print("CURRENT GATE THRESHOLDS (config.py)\n")
for k in ("CM_MIN_CORR", "CM_MIN_INVERSION_PCT", "STRESS_MAX_UNDERSTATED", "STRESS_MIN_RATIO",
          "MAX_DATA_AGE_DAYS", "MAX_CALIBRATION_AGE_DAYS"):
    print(f"  {k:28} = {getattr(config, k, '(not set)')}")

print("\n\nONE THRESHOLD TO CONCEDE BEFORE BEING ASKED:")
print(f"  STRESS_MAX_UNDERSTATED = {config.STRESS_MAX_UNDERSTATED}, and the actual count at 30d is 2.")
print("  The gate was set to ACCOMMODATE the two known misses (Feb 2018, Aug 2024).")
print("  It is a REGRESSION GUARD against a NEW failure - not an independent test")
print("  the model passes on its own merits. Say that first; do not be caught on it.")

CURRENT GATE THRESHOLDS (config.py)

  CM_MIN_CORR                  = 0.9
  CM_MIN_INVERSION_PCT         = 50.0
  STRESS_MAX_UNDERSTATED       = 2
  STRESS_MIN_RATIO             = 0.5
  MAX_DATA_AGE_DAYS            = 7
  MAX_CALIBRATION_AGE_DAYS     = 400


ONE THRESHOLD TO CONCEDE BEFORE BEING ASKED:
  STRESS_MAX_UNDERSTATED = 2, and the actual count at 30d is 2.
  The gate was set to ACCOMMODATE the two known misses (Feb 2018, Aug 2024).
  It is a REGRESSION GUARD against a NEW failure - not an independent test
  the model passes on its own merits. Say that first; do not be caught on it.


## Step 5 — reading a run folder

Every run writes a self-contained dated folder under `output/runs/`. Nothing is
overwritten, so any number ever reported can be traced back.

In [5]:
runs = sorted((REPO / "output" / "runs").glob("20*"))
print(f"{len(runs)} run folder(s) present\n")
for r in runs[-5:]:
    kind = "CALIBRATION" if r.name.endswith("calibrate") else "pricing"
    mf = r / "manifest.json"
    verdict = ""
    if mf.exists():
        m = json.loads(mf.read_text())
        verdict = "PASS" if m.get("all_gates_pass") else "FAIL"
    print(f"  {r.name:32} {kind:12} {verdict}")

print("\n\nWHAT IS IN A PRICING RUN FOLDER\n")
print("  report.txt            the full run, same as stdout")
print("  manifest.json         provenance - READ THIS FIRST")
print("  positions.csv         every position x every scenario")
print("  pnl_by_scenario.csv   the headline table")
print("  shocked_curves.csv    the curve under each scenario")
print("  book_input.csv        a copy of the book as supplied")
print("\nA CALIBRATION folder additionally keeps both parameter files and the three")
print("validation plots - the evidence for the numbers it produced.")

33 run folder(s) present

  20260919_112359_price            pricing      FAIL
  20260919_112441_price            pricing      PASS
  20260919_112442_price            pricing      PASS
  20260919_112443_price            pricing      PASS
  20260919_112444_price            pricing      FAIL


WHAT IS IN A PRICING RUN FOLDER

  report.txt            the full run, same as stdout
  manifest.json         provenance - READ THIS FIRST
  positions.csv         every position x every scenario
  pnl_by_scenario.csv   the headline table
  shocked_curves.csv    the curve under each scenario
  book_input.csv        a copy of the book as supplied

A CALIBRATION folder additionally keeps both parameter files and the three
validation plots - the evidence for the numbers it produced.


### The manifest is the reproducibility answer

**Two runs of the same book that disagree can only differ on two things:** the
calibration parameters, or the market-data date. Both are in the manifest, so diffing
two manifests answers "why did this number change?" immediately.

In [6]:
pricing = [r for r in runs if r.name.endswith("_price") and (r / "manifest.json").exists()]
if pricing:
    m = json.loads((pricing[-1] / "manifest.json").read_text())
    print(f"most recent pricing run: {m['run_id']}\n")
    for k in ("run_at", "asof", "book_file", "n_positions",
              "positions_priced_off_market", "all_gates_pass"):
        print(f"  {k:30} {m.get(k)}")
    print(f"  {'market data date':30} {m['market_data']['latest_curve_date']} "
          f"({m['market_data']['age_days']} days old)")
    vr = m["calibration"]["vix_response"]
    print(f"  {'calibration fit window':30} {vr['fit_window']}")
    print(f"  {'calibration params':30} {vr['params']}")
    print(f"  {'gates':30} {m['gates']}")
else:
    print("No pricing run yet. Run:  python price.py --book input/book_TEMPLATE.csv")

most recent pricing run: 20260919_112444_price

  run_at                         2026-09-19 11:24:44
  asof                           2026-09-18
  book_file                      C:\Users\vmc30\AppData\Local\Temp\pytest-of-vmc30\pytest-5\test_book_vols_gate_fails_on_m0\no_premiums.csv
  n_positions                    1
  positions_priced_off_market    0
  all_gates_pass                 False
  market data date               2026-09-18 (1 days old)
  calibration fit window         2004-03-29 -> 2026-09-18
  calibration params             {'beta_0': 197.51, 'k': 0.7586, 'lam': 0.209, 'beta_up_0': 68.2641, 'lam_up': 0.2179}
  gates                          {'calibration_fresh': True, 'data_fresh': True, 'book_vols': False}


## Step 6 — the daily routine

```
1.  A book arrives           ->  input/book_<date>.csv
                                 every position needs a `premium` (mid where available)

2.  python price.py --book input/book_<date>.csv

3.  Check the VERDICT block.  Exit 0 = usable.

4.  Results in output/runs/<run-id>/
```

### When a gate fails

| failure | cause | fix |
|---|---|---|
| `book_vols` | positions without premium/vol | add premiums; never put `0` in the vol column |
| `data_fresh` | no recent curve | let it pull from CBOE, or drop files in `data/daily_inputs/` |
| `calibration_fresh` | parameters over a year old | `python run.py` — and treat it as a reviewed event |

### The yearly routine

```
1.  python run.py
2.  Read report.txt section 8 (VERDICT) and section 5 (stress validation)
3.  Keep the output/runs/<id>_calibrate/ folder - it is the calibration of record
4.  Commit it
```

Recalibrate off-cycle if: the post-2012 drift warning widens materially; a new stress
episode occurs that the model would have understated (add it to `stress.EPISODES` and
re-validate); or the VIX futures market changes structurally.

## Step 7 — what runs where

The work machine is air-gapped. This is what that implies.

In [7]:
rows = [
    ("bootstrap_history.py", "ONCE, build machine", "internet", "loads 2004-now history"),
    ("run.py", "yearly, reviewed", "history on disk", "refits; needs no network if data is there"),
    ("price.py", "daily", "curve + params", "no network, no key, no account"),
]
print(pd.DataFrame(rows, columns=["entry point", "when", "needs", "notes"]).to_string(index=False))

print("\n\nAIR-GAP CONSTRAINTS, all satisfied:")
print("  * No API keys anywhere - no market-data key, no AI-assistant key")
print("  * No Bloomberg code in the project; the historical file is frozen and optional")
print("  * No network required: data/daily_inputs/ supplies today's rows from local CSVs")
print(f"  * DAILY_REFRESH = {config.DAILY_REFRESH}: tries the public files, continues without them")

print("\n\nIF NOTHING CAN REFRESH THE DATA:")
print("  the data_fresh gate FAILS the run once the curve is over")
print(f"  {config.MAX_DATA_AGE_DAYS} days old. That is the intended behaviour - a")
print("  loud failure rather than a quiet wrong number.")

         entry point                when           needs                                     notes
bootstrap_history.py ONCE, build machine        internet                    loads 2004-now history
              run.py    yearly, reviewed history on disk refits; needs no network if data is there
            price.py               daily  curve + params            no network, no key, no account


AIR-GAP CONSTRAINTS, all satisfied:
  * No API keys anywhere - no market-data key, no AI-assistant key
  * No Bloomberg code in the project; the historical file is frozen and optional
  * No network required: data/daily_inputs/ supplies today's rows from local CSVs
  * DAILY_REFRESH = True: tries the public files, continues without them


IF NOTHING CAN REFRESH THE DATA:
  the data_fresh gate FAILS the run once the curve is over
  7 days old. That is the intended behaviour - a
  loud failure rather than a quiet wrong number.


---

## What to take away

**Two entry points, two purposes.** `run.py` calibrates (yearly, minutes, reviewed);
`price.py` prices (daily, ~1 second, frozen parameters). The split exists because a model
that re-derives itself on every run cannot be validated.

**The model is ten numbers.** Everything in `data/raw/` exists to produce them. Pricing
needs only today's curve, spot VIX, the vol-of-vol level, and those ten numbers.

**SPX is not a pricing input.** The SPX move is a scenario you choose. `shock.py` reads
no SPX at all — which is why the Yahoo dependency is not a daily one.

**Everything fails loudly now.** Stale data, a missing calibration, a book without
premiums — each is a gate, each exits non-zero. That was not true before; the model used
to print ALL GATES PASS on weeks-old data.

**Every run is traceable.** `output/runs/<run-id>/manifest.json` records the parameters,
fit window, market-data date and every gate result. Two runs that disagree differ on the
calibration or the data date — nothing else.

**Concede the stress-gate threshold first.** It allows exactly 2 understated episodes per
tenor and the actual count is exactly 2. It is a regression guard, not an independent test.

**Code:** `run.py`, `price.py`, `vixshock/diagnostics.py`. Contracts:
`input/INPUT_CONTRACT.md`, `data/daily_inputs/README.md`, `output/runs/README.md`.

---

### Questions to test yourself

1. A colleague runs `price.py` and gets exit code 1 with `book_vols: FAIL`. What happened
   and what do they do?
2. Why is refitting on every run a governance problem, not just a speed problem?
3. The machine has no internet for two weeks. What happens on day 3, and why is that the
   correct behaviour?
4. Two runs of the same book a month apart give different P&L. How do you find out why?
5. Why does the daily path not need SPX data?

---

That is the last module. For what is *wrong* with the model and how big it is, read
`../FINDINGS.md`. For the open questions, `../../QUESTIONS_FOR_QUANT.md`.